## AdventureWorks Streaming Medallion (Local PySpark)

- Local-only PySpark (no Databricks, no Delta). Parquet in `spark-warehouse`.
- Sources: local MySQL AdventureWorks (OLTP), Mongo Atlas reviews, local CSV/JSON in `data/`.
- Flow: batch dims -> export fact to streaming JSON -> bronze (append) -> silver (dim joins) -> gold aggregates (complete).
- Replace credential placeholders before running.


In [ ]:
import os
import json
import math
import shutil
import datetime as dt
import pandas as pd
from pathlib import Path

import findspark
findspark.init()

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, input_file_name, asc, desc, count, avg, sum as spark_sum
from pyspark.sql.types import IntegerType, LongType, DateType

import sqlalchemy
from sqlalchemy import create_engine
import pymongo
import certifi

# Configurations for MySQL local and MongoDB Atlas
mysql = {
    "uid": "root",
    "pwd": "<YOUR_PASSWORD>",
    "hostname": "localhost",
    "src_db": "adventureworks",
    "dw_db": "adventureworks_dw"
}

mongo = {
    "user_name": "<YOUR_USERNAME>",
    "password": "<YOUR_PASSWORD>",
    "cluster_name": "<CLUSTER_NAME>",
    "cluster_subnet": "<CLUSTER_SUBNET>",
    "cluster_location": "atlas",  # or "local"
    "db_name": "adventureworks_docs",
    "review_coll": "product_review_aggregates"
}

BASE_DIR = Path(os.getcwd())
DATA_DIR = BASE_DIR / "data"
STREAM_DIR = DATA_DIR / "stream"
FACT_SALES_STREAM_DIR = STREAM_DIR / "fact_sales"

SQL_WAREHOUSE_DIR = Path(os.getcwd()) / "spark-warehouse"
DEST_DB = "adventureworks_dlh"
DB_DIR = SQL_WAREHOUSE_DIR / f"{DEST_DB}.db"

FACT_SALES_BRONZE = DB_DIR / "fact_sales" / "bronze"
FACT_SALES_SILVER = DB_DIR / "fact_sales" / "silver"


In [2]:
def get_mongo_client():
    if mongo["cluster_location"] == "atlas":
        uri = (
            f"mongodb+srv://{mongo['user_name']}:{mongo['password']}@"
            f"{mongo['cluster_name']}.{mongo['cluster_subnet']}.mongodb.net"
        )
        return pymongo.MongoClient(uri, tlsCAFile=certifi.where())
    return pymongo.MongoClient("mongodb://localhost:27017/")


def drop_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)

def ensure_dirs():
    for p in [DATA_DIR, STREAM_DIR, FACT_SALES_STREAM_DIR, FACT_SALES_BRONZE, FACT_SALES_SILVER]:
        p.mkdir(parents=True, exist_ok=True)


def get_spark():
    conf = (
        SparkConf()
        .setAppName("AdventureWorks Streaming (local)")
        .setMaster(f"local[{max(os.cpu_count()//2,1)}]")
        .set("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
        .set("spark.sql.streaming.schemaInference", "true")
        .set("spark.sql.warehouse.dir", str(SQL_WAREHOUSE_DIR))
        .set("spark.sql.shuffle.partitions", max(os.cpu_count(), 4))
    )
    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark


def get_mysql_engine(db: str):
    conn = f"mysql+pymysql://{mysql['uid']}:{mysql['pwd']}@{mysql['hostname']}/{db}"
    return create_engine(conn, pool_recycle=3600)


def export_fact_sales_to_stream_files(chunk_size: int = 5000):
    sql = """
    SELECT 
      d.SalesOrderID,
      d.SalesOrderDetailID,
      h.OrderDate,
      h.CustomerID,
      d.ProductID,
      d.OrderQty,
      d.UnitPrice,
      d.UnitPriceDiscount,
      d.LineTotal,
      h.SubTotal,
      h.TaxAmt,
      h.Freight,
      h.TotalDue
    FROM salesorderdetail d
    JOIN salesorderheader h ON h.SalesOrderID = d.SalesOrderID
    """
    eng = get_mysql_engine(mysql["src_db"])
    df = pd.read_sql(sql, eng)
    drop_dir(FACT_SALES_STREAM_DIR)
    FACT_SALES_STREAM_DIR.mkdir(parents=True, exist_ok=True)
    num_chunks = math.ceil(len(df) / chunk_size)
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(df))
        chunk = df.iloc[start:end]
        out_path = FACT_SALES_STREAM_DIR / f"fact_sales_{i+1:03d}.json"
        chunk.to_json(out_path, orient="records", lines=False, indent=2, date_format="iso")
    return len(df), num_chunks


def load_mongo_reviews_as_df():
    client = get_mongo_client()
    try:
        db = client[mongo["db_name"]]
        docs = list(db[mongo["review_coll"]].find({}))
        if not docs:
            return pd.DataFrame()
        for d in docs:
            d.pop("_id", None)
        return pd.DataFrame(docs)
    finally:
        client.close()


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        import time
        time.sleep(5)
    print(f"Stream processed {len(query.recentProgress)} batch(es)")


In [3]:
ensure_dirs()
spark = get_spark()
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DEST_DB}")
spark.sql(f"USE {DEST_DB}")
print(f"Warehouse: {SQL_WAREHOUSE_DIR}")


25/12/16 21:49:28 WARN Utils: Your hostname, Sebastians-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.2.163.251 instead (on interface en0)
25/12/16 21:49:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/16 21:49:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Warehouse: /Users/sebastian/ds-systems/ds2002-midterm/spark-warehouse


In [4]:
# Batch dimensions
from sqlalchemy import text

# Date dimension from OLTP min/max dates
sql_minmax = """
SELECT 
  MIN(DATE(OrderDate)) AS min_date,
  GREATEST(MAX(DATE(ShipDate)), MAX(DATE(DueDate))) AS max_date
FROM salesorderheader;
"""
mm = pd.read_sql(text(sql_minmax), get_mysql_engine(mysql["src_db"]))
start_date = pd.to_datetime(mm.loc[0, "min_date"]).date()
end_date = pd.to_datetime(mm.loc[0, "max_date"]).date()
if pd.isna(start_date) or pd.isna(end_date):
    start_date = dt.date(2000, 1, 1)
    end_date = dt.date(2010, 12, 31)

all_days = pd.date_range(start=start_date, end=end_date, freq="D")
df_date_pd = pd.DataFrame({"full_date": all_days})
df_date_pd["date_key"] = df_date_pd["full_date"].dt.strftime("%Y%m%d").astype(int)
df_date_pd["year"] = df_date_pd["full_date"].dt.year
df_date_pd["quarter"] = df_date_pd["full_date"].dt.quarter
df_date_pd["month"] = df_date_pd["full_date"].dt.month
df_date_pd["day"] = df_date_pd["full_date"].dt.day
df_date_pd["day_name"] = df_date_pd["full_date"].dt.day_name()
df_date_pd["month_name"] = df_date_pd["full_date"].dt.month_name()
df_date_pd["week_of_year"] = df_date_pd["full_date"].dt.isocalendar().week.astype(int)
df_date_pd["is_weekend"] = df_date_pd["day_name"].isin(["Saturday", "Sunday"])

# Idempotent drop + clean location
spark.sql(f"DROP TABLE IF EXISTS {DEST_DB}.dim_date")
drop_dir(DB_DIR / "dim_date")
spark.createDataFrame(df_date_pd).write.mode("overwrite").saveAsTable(f"{DEST_DB}.dim_date")

# Product dimension: MySQL product + subcategory/category + CSV attrs + Mongo reviews
sql_dim_product_base = """
SELECT 
  p.ProductID,
  p.Name AS product_name,
  p.ProductNumber,
  p.Color,
  p.ListPrice,
  p.ProductSubcategoryID,
  sc.Name AS subcategory_name,
  sc.ProductCategoryID,
  c.Name AS category_name
FROM product p
LEFT JOIN productsubcategory sc ON p.ProductSubcategoryID = sc.ProductSubcategoryID
LEFT JOIN productcategory c ON sc.ProductCategoryID = c.ProductCategoryID;
"""
prod_base_pd = pd.read_sql(text(sql_dim_product_base), get_mysql_engine(mysql["src_db"]))
attrs = pd.read_csv(DATA_DIR / "product_attributes.csv")
prod_enriched = prod_base_pd.merge(attrs, on="ProductID", how="left")
prod_enriched["OnlineOnly"] = prod_enriched["OnlineOnly"].fillna(False).astype(bool)
reviews_pd = load_mongo_reviews_as_df()
if not reviews_pd.empty:
    prod_enriched = prod_enriched.merge(reviews_pd, on="ProductID", how="left")

cols_product = [
    "ProductID", "product_name", "ProductNumber", "Color", "ListPrice",
    "subcategory_name", "category_name",
    "MarketingSegment", "BrandTier", "OnlineOnly", "Season", "LaunchYear",
    "num_reviews_sql", "avg_rating_sql"
]
prod_final = prod_enriched[cols_product]
# Idempotent drop + clean location
spark.sql(f"DROP TABLE IF EXISTS {DEST_DB}.dim_product")
drop_dir(DB_DIR / "dim_product")
spark.createDataFrame(prod_final).write.mode("overwrite").saveAsTable(f"{DEST_DB}.dim_product")

# Customer dimension: store + individual + territory
sql_store = """
SELECT c.CustomerID,
       c.AccountNumber,
       c.CustomerType,
       c.TerritoryID,
       s.Name AS store_name,
       NULL AS first_name,
       NULL AS last_name
FROM customer c
JOIN store s ON s.CustomerID = c.CustomerID
"""
sql_individual = """
SELECT c.CustomerID,
       c.AccountNumber,
       c.CustomerType,
       c.TerritoryID,
       NULL AS store_name,
       ct.FirstName AS first_name,
       ct.LastName  AS last_name
FROM customer c
JOIN individual i ON i.CustomerID = c.CustomerID
JOIN contact ct ON ct.ContactID = i.ContactID
"""
stores_pd = pd.read_sql(text(sql_store), get_mysql_engine(mysql["src_db"]))
inds_pd = pd.read_sql(text(sql_individual), get_mysql_engine(mysql["src_db"]))
cust_all = pd.concat([stores_pd, inds_pd], ignore_index=True)
terr_pd = pd.read_sql(text("SELECT TerritoryID, Name AS territory_name, CountryRegionCode, `Group` AS terr_group FROM salesterritory;"), get_mysql_engine(mysql["src_db"]))
cust_dim_pd = cust_all.merge(terr_pd, on="TerritoryID", how="left")
cols_cust = [
    "CustomerID", "AccountNumber", "CustomerType", "store_name", "first_name", "last_name",
    "TerritoryID", "territory_name", "CountryRegionCode", "terr_group"
]
# Idempotent drop + clean location
spark.sql(f"DROP TABLE IF EXISTS {DEST_DB}.dim_customer")
drop_dir(DB_DIR / "dim_customer")
spark.createDataFrame(cust_dim_pd[cols_cust]).write.mode("overwrite").saveAsTable(f"{DEST_DB}.dim_customer")


In [5]:
# Export fact_sales from MySQL to streaming JSON files
rows, files = export_fact_sales_to_stream_files(chunk_size=5000)
print(f"Exported {rows} rows into {files} streaming file(s) at {FACT_SALES_STREAM_DIR}")


Exported 121317 rows into 25 streaming file(s) at /Users/sebastian/ds-systems/ds2002-midterm/data/stream/fact_sales


In [6]:
# Bronze: stream fact_sales
drop_dir(FACT_SALES_BRONZE)
FACT_SALES_BRONZE.mkdir(parents=True, exist_ok=True)
bronze_checkpoint = FACT_SALES_BRONZE / "_checkpoint"

fact_bronze_stream = (
    spark.readStream
    .option("schemaLocation", str(FACT_SALES_BRONZE / "_schema"))
    .option("maxFilesPerTrigger", 1)
    .option("multiLine", "true")
    .json(str(FACT_SALES_STREAM_DIR))
)

fact_bronze_query = (
    fact_bronze_stream
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    .writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("fact_sales_bronze")
    .trigger(availableNow=True)
    .option("checkpointLocation", str(bronze_checkpoint))
    .option("compression", "snappy")
    .start(str(FACT_SALES_BRONZE))
)
print(f"Bronze query: {fact_bronze_query.id}")
fact_bronze_query.awaitTermination()
print(f"Bronze files: {len(list(FACT_SALES_BRONZE.glob('*.parquet')))}")


25/12/16 21:49:41 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
25/12/16 21:49:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Bronze query: 632a3036-7fcc-4a6f-99c0-885277569b4e
Bronze files: 25


In [7]:
# Silver: join dims
drop_dir(FACT_SALES_SILVER)
FACT_SALES_SILVER.mkdir(parents=True, exist_ok=True)
silver_checkpoint = FACT_SALES_SILVER / "_checkpoint"

# Load static dims
_dim_date = spark.table(f"{DEST_DB}.dim_date")
_dim_prod = spark.table(f"{DEST_DB}.dim_product")
_dim_cust = spark.table(f"{DEST_DB}.dim_customer")

f = (
    spark.readStream.format("parquet").load(str(FACT_SALES_BRONZE))
    .withColumn("date_key", col("OrderDate").cast(DateType()).cast("string"))
).alias("f")

p = _dim_prod.alias("p")
c = _dim_cust.alias("c")
d = _dim_date.alias("d")

fact_silver_joined = (
    f
    .join(p, col("f.ProductID") == col("p.ProductID"), "left")
    .join(c, col("f.CustomerID") == col("c.CustomerID"), "left")
    .join(d, col("f.date_key") == col("d.full_date").cast("string"), "left")
    .select(
        col("f.SalesOrderID").cast(LongType()).alias("SalesOrderID"),
        col("f.SalesOrderDetailID").cast(LongType()).alias("SalesOrderDetailID"),
        col("f.date_key").cast(IntegerType()).alias("date_key"),
        col("f.CustomerID").cast(LongType()).alias("CustomerID"),
        col("f.ProductID").cast(LongType()).alias("ProductID"),
        col("OrderQty"), col("UnitPrice"), col("UnitPriceDiscount"),
        col("LineTotal"), col("SubTotal"), col("TaxAmt"), col("Freight"), col("TotalDue"),
        col("p.category_name"), col("p.subcategory_name"), col("p.avg_rating_sql"),
        col("c.terr_group"), col("d.year")
    )
)

fact_silver_query = (
    fact_silver_joined.writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("fact_sales_silver")
    .trigger(availableNow=True)
    .option("checkpointLocation", str(silver_checkpoint))
    .option("compression", "snappy")
    .start(str(FACT_SALES_SILVER))
)
print(f"Silver query: {fact_silver_query.id}")
fact_silver_query.awaitTermination()


25/12/16 21:49:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Silver query: 2631f119-90c9-4f1a-a331-e4516ee6462d


In [9]:
# Gold: batch aggregates from silver parquet

# Quick sanity check that silver outputs exist
if (not FACT_SALES_SILVER.exists()) or (not list(FACT_SALES_SILVER.glob("*.parquet"))):
    raise ValueError(f"No silver parquet files found in {FACT_SALES_SILVER}. Run silver first.")

silver_batch = spark.read.format("parquet").load(str(FACT_SALES_SILVER))

# Reload dims to ensure availability
_dim_date = spark.table(f"{DEST_DB}.dim_date")
_dim_prod = spark.table(f"{DEST_DB}.dim_product")
_dim_cust = spark.table(f"{DEST_DB}.dim_customer")

date_small = _dim_date.select("date_key", "year").alias("d")
prod_small = _dim_prod.select("ProductID", "category_name", "product_name", "avg_rating_sql").alias("p")
cust_small = _dim_cust.select("CustomerID", "terr_group").alias("c")

# q1 total sales by year and category
q1 = (
    silver_batch.alias("f")
    .join(date_small, "date_key")
    .join(prod_small, "ProductID")
    .groupBy(col("d.year"), col("p.category_name"))
    .agg(spark_sum("LineTotal").alias("total_sales"))
    .select(col("d.year").alias("year"), col("p.category_name").alias("category_name"), col("total_sales"))
    .orderBy("year", "category_name")
)

# q2 avg discount by territory group and year
q2 = (
    silver_batch.alias("f")
    .join(date_small, "date_key")
    .join(cust_small, "CustomerID")
    .groupBy(col("d.year"), col("c.terr_group"))
    .agg(avg("UnitPriceDiscount").alias("avg_discount"))
    .select(col("d.year").alias("year"), col("c.terr_group").alias("terr_group"), col("avg_discount"))
    .orderBy("year", "terr_group")
)

# q3 top products by total sales and rating
q3 = (
    silver_batch.alias("f")
    .join(prod_small, "ProductID")
    .groupBy(col("p.product_name"), col("p.category_name"), col("p.avg_rating_sql"))
    .agg(spark_sum("LineTotal").alias("total_sales"))
    .select(
        col("p.product_name").alias("product_name"),
        col("p.category_name").alias("category_name"),
        col("p.avg_rating_sql").alias("avg_rating_sql"),
        col("total_sales")
    )
    .orderBy(desc("total_sales"))
    .limit(50)
)

for name, df in [
    ("fact_sales_by_category", q1),
    ("fact_sales_discount_by_terr", q2),
    ("fact_sales_top_products", q3),
]:
    spark.sql(f"DROP TABLE IF EXISTS {DEST_DB}.{name}")
    drop_dir(DB_DIR / name)
    df.write.mode("overwrite").saveAsTable(f"{DEST_DB}.{name}")

print("Gold tables materialized (batch).")


Gold tables materialized (batch).
